# MS_00 — Multi-Station EDA

Runs the same EDA figures as `00_EDA.ipynb` for every station in `STATIONS_TO_RUN`.
Checkpoint: skips a station if `outputs/{station}/figures/target_distribution.png` already exists.

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    DATA_PATH, ALL_STATIONS, MET_COLS, HYSPLIT_CATEGORICAL,
    WHO_24H_PM25, STATION_COORDS, get_station_paths,
)
from src.data_loader import load_data
from src.utils import ensure_dirs

sns.set_theme(style='whitegrid', palette='tab10')

# Override to run a subset, e.g. STATIONS_TO_RUN = ["MzWarChrosci"]
STATIONS_TO_RUN = ALL_STATIONS

print(f'Stations to process: {STATIONS_TO_RUN}')

Stations to process: ['MzWarChrosci', 'MzOtwoBrzozo', 'MzWarWokalna', 'MzWarAlNiepo', 'MzLegZegrzyn', 'MzPiasPulask', 'MzWarBajkowa']


In [2]:
# Load full dataset once — all EDA reads from the same CSV
df_full = load_data(DATA_PATH)
print(f'Loaded: {df_full.shape} | {df_full.index.min()} → {df_full.index.max()}')

06:37:24 | src.data_loader | INFO | Loading data from D:\MOJE\DATA_SCIENCE\ML_WARSAW_AQI_TOY\warsaw_aq_forecast\data\raw\FINAL_merged_PM25_1g_all_seasons.csv
06:37:25 | src.data_loader | INFO | Missing values per column:
MzOtwoBrzozo    1084
MzWarWokalna    2028
MzWarAlNiepo     575
MzLegZegrzyn    1511
MzPiasPulask    1002
MzWarChrosci     416
MzWarBajkowa     636
dir_48            48
dir_24            48
06:37:25 | src.data_loader | INFO | Loaded 52608 rows, 27 columns, range 2019-01-01 00:00:00 → 2024-12-31 23:00:00


Loaded: (52608, 27) | 2019-01-01 00:00:00 → 2024-12-31 23:00:00


In [3]:
n = len(STATIONS_TO_RUN)

for station in STATIONS_TO_RUN:
    paths = get_station_paths(station)
    checkpoint = paths['figures'] / 'target_distribution.png'

    if checkpoint.exists():
        print(f'[SKIP] {station} — target_distribution.png already exists')
        continue

    if station not in df_full.columns:
        print(f'[WARN] {station} — column not found in CSV, skipping')
        continue

    print(f'\n>>> {station}')
    ensure_dirs(paths['figures'])
    fig_dir = paths['figures']

    # ── Target distribution ───────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].hist(df_full[station].dropna(), bins=80, color='steelblue', edgecolor='white', alpha=0.8)
    axes[0].axvline(WHO_24H_PM25, color='green', linestyle='--', label=f'WHO {WHO_24H_PM25} µg/m³')
    axes[0].set_xlabel('PM2.5 (µg/m³)')
    axes[0].set_title(f'{station} — Distribution (raw)')
    axes[0].legend()

    sns.kdeplot(np.log1p(df_full[station].dropna()), ax=axes[1], fill=True, color='steelblue')
    axes[1].set_xlabel('log1p(PM2.5)')
    axes[1].set_title('Distribution (log1p scale)')

    monthly = df_full[station].resample('ME').mean()
    axes[2].plot(monthly.index, monthly.values, color='steelblue', linewidth=1.5)
    axes[2].axhline(WHO_24H_PM25, color='green', linestyle='--')
    axes[2].set_title('Monthly Mean PM2.5')
    axes[2].set_ylabel('µg/m³')

    fig.tight_layout()
    fig.savefig(fig_dir / 'target_distribution.png', bbox_inches='tight')
    plt.close(fig)

    # ── Temporal patterns ─────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    temp_df = df_full[[station]].copy()
    temp_df['hour'] = temp_df.index.hour
    temp_df['month'] = temp_df.index.month
    temp_df['dow'] = temp_df.index.day_of_week

    temp_df.boxplot(column=station, by='hour', ax=axes[0], showfliers=False)
    axes[0].set_title('By Hour of Day')
    axes[0].set_xlabel('Hour')
    axes[0].set_ylabel('PM2.5 (µg/m³)')

    temp_df.boxplot(column=station, by='month', ax=axes[1], showfliers=False)
    axes[1].set_title('By Month')
    axes[1].set_xlabel('Month')

    temp_df.boxplot(column=station, by='dow', ax=axes[2], showfliers=False)
    axes[2].set_title('By Day of Week (0=Mon)')
    axes[2].set_xlabel('Day of Week')

    for ax in axes:
        ax.axhline(WHO_24H_PM25, color='green', linestyle='--', linewidth=0.8)

    fig.suptitle('')
    fig.tight_layout()
    fig.savefig(fig_dir / 'temporal_patterns.png', bbox_inches='tight')
    plt.close(fig)

    # ── Correlation heatmap ───────────────────────────────────────────
    corr_cols = [station] + [c for c in MET_COLS if c in df_full.columns]
    corr = df_full[corr_cols].corr()

    fig, ax = plt.subplots(figsize=(10, 8))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                vmin=-1, vmax=1, ax=ax, linewidths=0.5)
    ax.set_title(f'{station} — Correlation: PM2.5 × Meteorological Variables')
    fig.tight_layout()
    fig.savefig(fig_dir / 'correlation_heatmap.png', bbox_inches='tight')
    plt.close(fig)

    # ── BLH analysis ─────────────────────────────────────────────────
    if 'blh' in df_full.columns:
        df_blh = df_full[[station, 'blh']].dropna().copy()
        df_blh['season'] = df_blh.index.month.map(
            lambda m: 'Winter' if m in [12, 1, 2] else
                      'Spring' if m in [3, 4, 5] else
                      'Summer' if m in [6, 7, 8] else 'Autumn'
        )

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        palette = {'Winter': '#2c7bb6', 'Spring': '#1a9641', 'Summer': '#fdae61', 'Autumn': '#d7191c'}

        for season, grp in df_blh.groupby('season'):
            sample = grp.sample(min(2000, len(grp)), random_state=42)
            axes[0].scatter(sample['blh'], sample[station], alpha=0.15, s=8,
                            color=palette.get(season, 'grey'), label=season)
        axes[0].set_xlabel('BLH (m)')
        axes[0].set_ylabel('PM2.5 (µg/m³)')
        axes[0].set_title(f'{station} — BLH vs PM2.5 by Season')
        axes[0].legend(markerscale=3)
        axes[0].set_xlim(0, 3000)

        ep = df_full.loc['2022-01-01':'2022-01-14', [station, 'blh']].dropna()
        ax2 = axes[1]
        ax2b = ax2.twinx()
        ax2.plot(ep.index, ep[station], color='#d7191c', label='PM2.5', linewidth=1.5)
        ax2b.plot(ep.index, ep['blh'], color='#2c7bb6', label='BLH', linewidth=1.5, linestyle='--')
        ax2.set_ylabel('PM2.5 (µg/m³)', color='#d7191c')
        ax2b.set_ylabel('BLH (m)', color='#2c7bb6')
        ax2.set_title('PM2.5 vs BLH — Jan 2022 Episode')
        lines1, labs1 = ax2.get_legend_handles_labels()
        lines2, labs2 = ax2b.get_legend_handles_labels()
        ax2.legend(lines1 + lines2, labs1 + labs2)

        fig.tight_layout()
        fig.savefig(fig_dir / 'blh_analysis.png', bbox_inches='tight')
        plt.close(fig)

    # ── HYSPLIT direction analysis ────────────────────────────────────
    if 'dir_24' in df_full.columns:
        dir_pm25 = df_full[[station, 'dir_24']].dropna()
        dir_means = dir_pm25.groupby('dir_24')[station].mean().sort_values(ascending=False)
        dir_counts = dir_pm25.groupby('dir_24')[station].count()

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        dir_means.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
        axes[0].axhline(WHO_24H_PM25, color='green', linestyle='--')
        axes[0].set_title(f'{station} — Mean PM2.5 by HYSPLIT 24h Direction')
        axes[0].set_ylabel('Mean PM2.5 (µg/m³)')

        dir_counts.plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
        axes[1].set_title('Count by Direction')
        axes[1].set_ylabel('Count (hours)')

        fig.tight_layout()
        fig.savefig(fig_dir / 'hysplit_analysis.png', bbox_inches='tight')
        plt.close(fig)

    # ── Inter-station correlation ─────────────────────────────────────
    present = [s for s in ALL_STATIONS if s in df_full.columns]
    station_corr = df_full[present].corr()

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(station_corr, annot=True, fmt='.2f', cmap='YlOrRd',
                vmin=0, vmax=1, ax=ax, linewidths=0.5)
    ax.set_title(f'{station} — Inter-Station PM2.5 Correlation')
    fig.tight_layout()
    fig.savefig(fig_dir / 'station_correlation.png', bbox_inches='tight')
    plt.close(fig)

    print(f'  Figures saved to {fig_dir}')

print('\nAll stations done.')


>>> MzWarChrosci
  Figures saved to D:\MOJE\DATA_SCIENCE\ML_WARSAW_AQI_TOY\warsaw_aq_forecast\outputs\MzWarChrosci\figures

>>> MzOtwoBrzozo
  Figures saved to D:\MOJE\DATA_SCIENCE\ML_WARSAW_AQI_TOY\warsaw_aq_forecast\outputs\MzOtwoBrzozo\figures

>>> MzWarWokalna
  Figures saved to D:\MOJE\DATA_SCIENCE\ML_WARSAW_AQI_TOY\warsaw_aq_forecast\outputs\MzWarWokalna\figures

>>> MzWarAlNiepo
  Figures saved to D:\MOJE\DATA_SCIENCE\ML_WARSAW_AQI_TOY\warsaw_aq_forecast\outputs\MzWarAlNiepo\figures

>>> MzLegZegrzyn
  Figures saved to D:\MOJE\DATA_SCIENCE\ML_WARSAW_AQI_TOY\warsaw_aq_forecast\outputs\MzLegZegrzyn\figures

>>> MzPiasPulask
  Figures saved to D:\MOJE\DATA_SCIENCE\ML_WARSAW_AQI_TOY\warsaw_aq_forecast\outputs\MzPiasPulask\figures

>>> MzWarBajkowa
  Figures saved to D:\MOJE\DATA_SCIENCE\ML_WARSAW_AQI_TOY\warsaw_aq_forecast\outputs\MzWarBajkowa\figures

All stations done.
